# N-Pulse — continue YOLO26s training on Colab GPU

## Goal

Continue training the published **YOLO26s Open Images + D405 v1** checkpoint on the validated `openimages6_d405_v1` dataset. The notebook verifies every input before starting and saves every epoch to Google Drive.

### Upload these files to Google Drive first

```text
MyDrive/npulse/
├── training_code.zip
├── data/
│   ├── openimages6_d405_v1.zip
│   └── openimages6_d405_v1.yaml       # optional; generated automatically if absent
└── models/
    └── yolo26s-mix-openimages-d405-v1.0.0/
        └── best.pt
```

Create the ZIP files locally:

```bash
bash training/colab/make_code_zip.sh
bash training/colab/make_data_zip.sh openimages6_d405_v1
```

Before running, choose **Runtime → Change runtime type → GPU**. Colab GPU type and availability vary.

In [ ]:
# 1. Parameters — edit only this cell for a normal run
RUN_NAME = "b3_yolo26s_mix_continue_lr2e4"
DATASET_NAME = "openimages6_d405_v1"
DATA_ZIPS = [f"{DATASET_NAME}.zip"]
START_MODEL_DRIVE = "models/yolo26s-mix-openimages-d405-v1.0.0/best.pt"
DATASET_YAML_DRIVE = f"data/{DATASET_NAME}.yaml"  # optional

EPOCHS = 50
IMAGE_SIZE = 416
BATCH_SIZE = 32
PATIENCE = 50
OPTIMIZER = "AdamW"
LEARNING_RATE = 0.0002
SEED = 0
FRACTION = 1.0

DRIVE_ROOT = "/content/drive/MyDrive/npulse"
REFRESH_DATA = False       # True: unzip dataset again
ALLOW_EXISTING_RUN = False # False prevents accidental run-name reuse

print({
    "run": RUN_NAME,
    "dataset": DATASET_NAME,
    "start_model": START_MODEL_DRIVE,
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "optimizer": OPTIMIZER,
    "lr0": LEARNING_RATE,
})

## Setup

The next cells install the runtime packages, mount Drive and verify that a CUDA GPU is actually available.

In [ ]:
# 2. Install the same training stack used by the project
%pip install -q "ultralytics>=8.4" pyyaml pandas

In [ ]:
# 3. Mount Google Drive and define paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

DRIVE = Path(DRIVE_ROOT)
REPO = Path("/content/ARM-SOFT-Grasping")
DATASETS = REPO / "training" / "datasets"
RUNS_DRIVE = DRIVE / "runs"
MODEL_PATH = DRIVE / START_MODEL_DRIVE
DRIVE_YAML = DRIVE / DATASET_YAML_DRIVE

print("Drive:", DRIVE)
print("Repository:", REPO)

In [ ]:
# 4. Require a real NVIDIA GPU before training
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. In Colab choose Runtime > Change runtime type > GPU, "
        "then reconnect and run from the top."
    )

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"], check=False)

## Prepare code and data

This section fails early with a clear list if an upload is missing. Re-running it is safe.

In [ ]:
# 5. Verify the uploaded inputs before changing the Colab workspace
required = [DRIVE / "training_code.zip", MODEL_PATH]
required += [DRIVE / "data" / archive for archive in DATA_ZIPS]
missing = [path for path in required if not path.is_file()]

for path in required:
    print(("OK     " if path.is_file() else "MISSING"), path)
if missing:
    raise FileNotFoundError("Upload the missing files to Drive: " + ", ".join(map(str, missing)))

print("Start model size (MB):", round(MODEL_PATH.stat().st_size / 1e6, 1))

# Confirm that the uploaded checkpoint is the intended six-class detector.
from ultralytics import YOLO
probe = YOLO(str(MODEL_PATH))
expected_classes = ["ball", "can", "box", "bottle", "pen_pencil", "door_handle"]
if list(probe.names.values()) != expected_classes:
    raise RuntimeError(f"Unexpected model classes: {probe.names}")
print("Checkpoint task:", probe.task)
print("Checkpoint classes:", probe.names)
del probe

In [ ]:
# 6. Extract the latest training code
if REPO.exists():
    shutil.rmtree(REPO)
REPO.mkdir(parents=True)
with zipfile.ZipFile(DRIVE / "training_code.zip") as archive:
    archive.extractall(REPO)

required_code = [REPO / "training" / "train.py", REPO / "training" / "data"]
if not all(path.exists() for path in required_code):
    raise RuntimeError("training_code.zip has the wrong structure; rebuild it from the repository root")
print("Code ready:", REPO / "training" / "train.py")

In [ ]:
# 7. Extract the versioned YOLO dataset
DATASETS.mkdir(parents=True, exist_ok=True)
dataset_root = DATASETS / DATASET_NAME

if REFRESH_DATA and dataset_root.exists():
    shutil.rmtree(dataset_root)
if not dataset_root.exists():
    for archive_name in DATA_ZIPS:
        archive_path = DRIVE / "data" / archive_name
        print("Extracting", archive_path.name)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(DATASETS)
else:
    print("Reusing extracted dataset:", dataset_root)

if not (dataset_root / "images" / "train").is_dir() or not (dataset_root / "labels" / "train").is_dir():
    raise RuntimeError(f"Dataset ZIP has the wrong structure: {dataset_root}")

In [ ]:
# 8. Copy the generated YAML, or recreate the same six-class configuration
import yaml

data_yaml = REPO / "training" / "data" / f"{DATASET_NAME}.yaml"
data_yaml.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_YAML.is_file():
    shutil.copy2(DRIVE_YAML, data_yaml)
    print("Copied dataset YAML from Drive")
else:
    names = {
        0: "ball",
        1: "can",
        2: "box",
        3: "bottle",
        4: "pen_pencil",
        5: "door_handle",
    }
    payload = {
        "path": str(dataset_root.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": names,
    }
    data_yaml.write_text(yaml.safe_dump(payload, sort_keys=False))
    print("Generated dataset YAML because the optional Drive YAML was absent")

config = yaml.safe_load(data_yaml.read_text())
print(data_yaml.read_text())
if list(config["names"].values()) != ["ball", "can", "box", "bottle", "pen_pencil", "door_handle"]:
    raise RuntimeError("Dataset class order is not the fixed six-class order")

In [ ]:
# 9. Confirm image/label counts and provenance
summary = {}
for split in ("train", "val", "test"):
    images = [p for p in (dataset_root / "images" / split).glob("*") if p.is_file()]
    labels = list((dataset_root / "labels" / split).glob("*.txt"))
    summary[split] = {"images": len(images), "labels": len(labels)}
    if not images or len(images) != len(labels):
        raise RuntimeError(f"Invalid {split} split: {summary[split]}")
print(json.dumps(summary, indent=2))

manifest = dataset_root / "dataset_manifest.json"
validation = dataset_root / "validation_report.json"
if manifest.is_file():
    print("Manifest:", json.loads(manifest.read_text()).get("dataset_name"))
if validation.is_file():
    report = json.loads(validation.read_text())
    print("Validated:", report.get("valid"), "errors:", len(report.get("errors", [])))
    if not report.get("valid", False):
        raise RuntimeError("The packaged dataset validation report is not valid")

## Train

The run directory is linked to Drive before training, so `best.pt`, `last.pt`, logs and plots survive a Colab disconnect.

In [ ]:
# 10. Connect training outputs to Drive and protect existing runs
RUNS_DRIVE.mkdir(parents=True, exist_ok=True)
detect = REPO / "training" / "runs" / "detect"
detect.parent.mkdir(parents=True, exist_ok=True)
if detect.is_symlink():
    detect.unlink()
elif detect.exists():
    shutil.rmtree(detect)
detect.symlink_to(RUNS_DRIVE, target_is_directory=True)

run_dir = RUNS_DRIVE / RUN_NAME
if run_dir.exists() and not ALLOW_EXISTING_RUN:
    raise FileExistsError(
        f"Run already exists: {run_dir}. Choose a new RUN_NAME. "
        "To resume, use the resume cell at the end instead of overwriting it."
    )
print("Outputs will be saved to:", run_dir)

In [ ]:
# 11. Start training with explicit, reproducible arguments
command = [
    sys.executable,
    str(REPO / "training" / "train.py"),
    "--device", "0",
    "--name", RUN_NAME,
    "--model", str(MODEL_PATH),
    "--data", str(data_yaml),
    "--epochs", str(EPOCHS),
    "--imgsz", str(IMAGE_SIZE),
    "--batch", str(BATCH_SIZE),
    "--patience", str(PATIENCE),
    "--optimizer", OPTIMIZER,
    "--lr0", str(LEARNING_RATE),
    "--seed", str(SEED),
    "--fraction", str(FRACTION),
]
print("Command:")
print(" ".join(command))
subprocess.run(command, cwd=REPO, check=True)

## Results

This reads the actual saved `args.yaml` and metrics rather than relying on the intended parameter cell.

In [ ]:
# 12. Show the saved configuration, best epoch and model files
import pandas as pd
from IPython.display import Image, display

run = RUNS_DRIVE / RUN_NAME
args_path = run / "args.yaml"
results_path = run / "results.csv"
weights = run / "weights"

print("Run:", run)
print("Weights:", sorted(p.name for p in weights.glob("*.pt")))
print("Saved arguments:")
print(args_path.read_text())

if (run / "results.png").is_file():
    display(Image(filename=str(run / "results.png"), width=1000))
if results_path.is_file():
    frame = pd.read_csv(results_path)
    frame.columns = [column.strip() for column in frame.columns]
    columns = [
        "epoch",
        "metrics/precision(B)",
        "metrics/recall(B)",
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)",
    ]
    print("Last epochs:")
    print(frame[columns].tail(3).to_string(index=False))
    best_index = frame["metrics/mAP50(B)"].idxmax()
    print()
    print("Best validation mAP50 epoch:")
    print(frame.loc[[best_index], columns].to_string(index=False))

## Resume after a disconnect

Run this only when `last.pt` exists and the original training stopped before the requested epochs.

In [ ]:
# 13. Optional resume — uncomment only after an interrupted run
# from ultralytics import YOLO
# last = RUNS_DRIVE / RUN_NAME / "weights" / "last.pt"
# if not last.is_file():
#     raise FileNotFoundError(last)
# YOLO(str(last)).train(resume=True)

## Next steps

1. Download the completed Drive run into `training/runs/detect/<RUN_NAME>/` locally.
2. Evaluate `best.pt` with `training/eval.py` on the fixed validation and test splits.
3. Compare it with the published catalog model before promoting a new model version.